In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

from pathlib import Path

# Make figures folder inside pdata directory
fig_dir = Path(pdata_root) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Convert to string for R compatibility
fig_dir_str = str(fig_dir)
fig_dir

Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (103980, 40)
Valid windows: (101861, 40)


PosixPath('/mnt/pdata/Classical_Conditioning/figures')

In [4]:
valid_windows

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,rig_day_from_date,rig_session_number,phase_day_from_date,phase_session_number,rig_session_start_min,phase_session_start_min,recording_duration_min,anchor_session_time_min,anchor_rig_time_min,anchor_phase_time_min
2,NML_04,2026_01_12,air_training,0,air_on,main,0.000000,post,1.0,air_on_post_1s,...,17,17,1,1,363.306667,0.000000,21.815,0.000000,363.306667,0.000000
4,NML_04,2026_01_12,air_training,0,pseudo_tone_off,pseudo,2.000000,post,1.0,pseudo_tone_off_post_1s,...,17,17,1,1,363.306667,0.000000,21.815,0.033333,363.340000,0.033333
5,NML_04,2026_01_12,air_training,0,pseudo_tone_off,pseudo,2.000000,pre,1.0,pseudo_tone_off_pre_1s,...,17,17,1,1,363.306667,0.000000,21.815,0.033333,363.340000,0.033333
6,NML_04,2026_01_12,air_training,0,air_on_mid,middle,2.435200,post,1.0,air_on_mid_post_1s,...,17,17,1,1,363.306667,0.000000,21.815,0.040587,363.347253,0.040587
7,NML_04,2026_01_12,air_training,0,air_on_mid,middle,2.435200,pre,1.0,air_on_mid_pre_1s,...,17,17,1,1,363.306667,0.000000,21.815,0.040587,363.347253,0.040587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103975,NML_08,2026_03_21,tone_air_training,48,air_on_mid,middle,1219.938477,pre,1.0,air_on_mid_pre_1s,...,48,47,10,10,965.440000,186.473333,20.545,20.332308,985.772308,206.805641
103976,NML_08,2026_03_21,tone_air_training,48,air_off,main,1223.568604,post,1.0,air_off_post_1s,...,48,47,10,10,965.440000,186.473333,20.545,20.392810,985.832810,206.866143
103977,NML_08,2026_03_21,tone_air_training,48,air_off,main,1223.568604,pre,1.0,air_off_pre_1s,...,48,47,10,10,965.440000,186.473333,20.545,20.392810,985.832810,206.866143
103978,NML_08,2026_03_21,tone_air_training,48,air_off_mid,middle,1231.068604,post,1.0,air_off_mid_post_1s,...,48,47,10,10,965.440000,186.473333,20.545,20.517810,985.957810,206.991143


In [3]:

from src.proc.behavior_metrics import compute_encoder_metrics_for_windows


encoder_epoch_df = compute_encoder_metrics_for_windows(
    valid_windows,
    speed_thresh=0.5
)

encoder_epoch_df.head()

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,max_speed_net_cms,distance_path_cm,distance_net_cm,net_direction_bias,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net_movement,dominant_locomotor_state
0,NML_04,2026_01_12,air_training,0,air_off_mid,middle,12.4134,post,1.0,air_off_mid_post_1s,...,9.550442,6.258053,6.258053,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward
1,NML_04,2026_01_12,air_training,0,air_off_mid,middle,12.4134,pre,1.0,air_off_mid_pre_1s,...,9.801769,4.297699,4.297699,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward
2,NML_04,2026_01_12,air_training,0,air_off,main,4.8704,post,1.0,air_off_post_1s,...,11.561061,6.182654,6.182654,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward
3,NML_04,2026_01_12,air_training,0,air_off,main,4.8704,pre,1.0,air_off_pre_1s,...,9.047787,4.146902,4.146902,1.000000,0.0156,0.9844,0.9844,0.0000,0.0,forward
4,NML_04,2026_01_12,air_training,0,air_on_mid,middle,2.4352,post,1.0,air_on_mid_post_1s,...,4.775221,2.161416,1.357168,0.636592,0.0814,0.9186,0.6674,0.2512,0.0,forward


In [6]:
cache_dir = Path(pdata_root) / "_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

encoder_metrics_file = cache_dir / "behavior_epoch_metrics.h5"

encoder_epoch_df.to_hdf(
    encoder_metrics_file,
    key="encoder/prepost_1s_speedThresh_1cms",
    mode="w",
    format="table"
)

print("Saved:", encoder_metrics_file)

Saved: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_metrics.h5
